In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("GPU Devices:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.10.0
Num GPUs Available: 1
GPU Devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
import os


In [ ]:
data_dir = "dataset"  # No /content/ or Colab-specific paths
batch_size = 32
img_height = 224
img_width = 224

# Load training and validation datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Check class names
class_names = train_ds.class_names
print("Classes:", class_names)

# Optimize performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 124486 files belonging to 4 classes.
Using 99589 files for training.
Found 124486 files belonging to 4 classes.
Using 24897 files for validation.
Classes: ['Flowering', 'Harvest', 'Seedling', 'Vegetative']


In [ ]:
# --------------------------------------------
# 🔹 Build EfficientNetB0 Model
# --------------------------------------------
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img_height, img_width, 3))
base_model.trainable = False  # Freeze the base model for feature extraction

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(len(class_names), activation='softmax')(x)

# Define full model
model = Model(inputs=base_model.input, outputs=predictions)

# --------------------------------------------
# 🔹 Compile the Model
# --------------------------------------------
model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --------------------------------------------
# 🔹 Train the Model for 20 Epochs
# --------------------------------------------
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


Epoch 1/20


In [ ]:
# Continue training the same model from epoch 21 to 50
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,        # Train up to epoch 50
    initial_epoch=20  # Start from epoch 21
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Get predictions and true labels
y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Step 2: Create confusion matrix (manual with NumPy)
num_classes = len(class_names)
confusion_mtx = np.zeros((num_classes, num_classes), dtype=int)

for true, pred in zip(y_true, y_pred):
    confusion_mtx[true][pred] += 1

print("Confusion Matrix:")
print(confusion_mtx)

# Step 3: Plot confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(confusion_mtx, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

# Annotate cells
thresh = confusion_mtx.max() / 2.
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, str(confusion_mtx[i, j]),
                 ha="center", va="center",
                 color="white" if confusion_mtx[i, j] > thresh else "black")

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()